# Lab 03 - Data Preprocessing: Impact of AI on Students

        Source dataset: `Datasets/Impact of AI on Students/ai_student_impact_dataset.csv`

        This notebook adapts the class lab pattern to the student-impact dataset. The source file is never modified.

        ## Lab concepts used

        - Validate schema and bounds.
- Remove identifiers and prevent target leakage.
- Encode, scale, and split data through a pipeline.

        Interpretation is predictive and associative only. The Kaggle source does not document how the records were collected or whether they represent observed students.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in [Path.cwd(), *Path.cwd().parents]:
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative} from {Path.cwd()}")

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns from {DATA_PATH}")

In [ ]:
IDENTIFIER = "Student_ID"
OUTCOMES = ["Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"]
EARLY_RISK_FEATURES = [
    "Major_Category", "Year_of_Study", "Pre_Semester_GPA",
    "Weekly_GenAI_Hours", "Primary_Use_Case",
    "Prompt_Engineering_Skill", "Tool_Diversity", "Paid_Subscription",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Institutional_Policy",
]
EXPANDED_FEATURES = EARLY_RISK_FEATURES + ["Anxiety_Level_During_Exams"]

df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)

assert IDENTIFIER not in EARLY_RISK_FEATURES
assert not set(OUTCOMES).intersection(EARLY_RISK_FEATURES)
print("Leakage policy ready. Primary burnout model excludes anxiety and all post-semester outcomes.")

## Quality, boundaries, and target audit

In [ ]:
expected_ranges = {
    "Pre_Semester_GPA": (1, 4),
    "Post_Semester_GPA": (1, 4),
    "Weekly_GenAI_Hours": (0, 40),
    "Tool_Diversity": (1, 5),
    "Perceived_AI_Dependency": (1, 10),
    "Anxiety_Level_During_Exams": (1, 10),
    "Skill_Retention_Score": (0, 100),
}
boundary_checks = []
for column, (lower, upper) in expected_ranges.items():
    invalid = (~df[column].between(lower, upper)).sum()
    boundary_checks.append({"column": column, "invalid_rows": int(invalid)})
display(pd.DataFrame(boundary_checks))
print("Missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("GPA-decline share:", f"{df['GPA_Declined'].mean():.2%}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def make_preprocessor(frame, scale_numeric=True):
    categorical = [
        column for column in frame.columns
        if pd.api.types.is_string_dtype(frame[column])
        or pd.api.types.is_bool_dtype(frame[column])
    ]
    numeric = [column for column in frame.columns if column not in categorical]
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical,
            ),
        ]
    )

## Leakage-safe train/test transformation

In [ ]:
from sklearn.model_selection import train_test_split

X = df[EARLY_RISK_FEATURES]
y = df["Burnout_Risk_Level"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
preprocessor = make_preprocessor(X_train)
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("Raw training shape:", X_train.shape)
print("Transformed training shape:", X_train_ready.shape)
print("Transformed test shape:", X_test_ready.shape)

## What was learned from Lab 3

The dataset needs no missing-value repair, so preprocessing focuses on legitimate schema validation, feature/target separation, identifier removal, categorical encoding, scaling, imbalance awareness, and fitting transformations only on training data.